[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesamKhallaf73/pytorch-ground-up-notebooks/blob/main/notebooks/L18.ipynb)

# Lecture 18 — Metrics That Matter & Visualization

*Measure what your model actually does — then look at it.*

**PyTorch from Ground Up · Volume I · Chapters 32–33** — companion notebook.

**How to use it:** run the cells from top to bottom. Before each code cell, *predict* what it will print — then run it and compare with the expected output (click ▸ to open it). Every exercise from the lecture is at the end.

By the end you will be able to:
- Explain why accuracy can lie, and read a confusion matrix.
- Compute precision, recall, F1 and specificity — and choose by what a mistake costs.
- Use macro vs weighted averaging and the four regression metrics.
- Explain thresholds, the ROC curve and AUC — and when to prefer precision-recall.
- Write an evaluation pipeline that scores every batch.
- Use the core plots: loss curves, predicted-vs-actual, residuals, decision boundaries, image grids, confusion heatmaps, filters, feature maps and embeddings.

In [ ]:
# This lecture uses torchmetrics, scikit-learn — installed here if missing (Colab doesn't ship them)
import importlib.util, subprocess, sys
for pkg, mod in [('torchmetrics', 'torchmetrics'), ('scikit-learn', 'sklearn')]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print('PyTorch', torch.__version__)

## 01 · 95% accurate — and useless: the confusion matrix

95% accurate — and possibly **useless**. Accuracy is a trap. On imbalanced data a model can say “negative” to everything and still score 95%. You need metrics that show what it actually does — and they all come from four counts.

<sub>Source: lecture slides 3–7, 15</sub>

🧪 *Interactive lab in the app:* **Four counts, five metrics** — try it there to see this idea move.

> ⚠️ **Watch out — Never accuracy alone**  
> Do not report accuracy on an imbalanced dataset without also reporting precision, recall or F1. A model that predicts the majority class for everything can score high accuracy and be completely useless.
>
> Always visualize the confusion matrix.

## 02 · Multiclass averaging and regression metrics

More than two classes: score each class, then average — **`macro`** treats classes equally, **`weighted`** by frequency. Use macro when rare classes matter as much as common ones. Here per-class F1 is 1.0, 0.5, 0.0: macro → 0.5, weighted (counts 2, 2, 1) → 0.6.

<sub>Source: lecture slides 8–14</sub>

**`metrics.py`** — predict the output, then run.

In [ ]:
from torchmetrics import Accuracy, Precision, Recall, F1Score
# Multiclass (3 classes)
preds = torch.tensor([0, 1, 2, 0, 1])
target = torch.tensor([0, 1, 1, 0, 2])
accuracy = Accuracy(task="multiclass", num_classes=3)
print(accuracy(preds, target))             # Fraction correct
# Macro-average (treat each class equally)
f1_macro = F1Score(task="multiclass", num_classes=3,
                   average="macro")
print(f1_macro(preds, target))
# Weighted-average (weight by class frequency)
f1_wtd = F1Score(task="multiclass", num_classes=3,
                 average="weighted")
print(f1_wtd(preds, target))

<details><summary>▸ Expected output</summary>

```
tensor(0.6000)
tensor(0.5000)
tensor(0.6000)
```
</details>

## 03 · Thresholds and ROC-AUC

Classifiers output **probabilities**; you choose the **threshold** that turns them into yes or no. Move the threshold and precision and recall trade places.

<sub>Source: lecture slides 16–20</sub>

🧪 *Interactive lab in the app:* **Slide the threshold** — try it there to see this idea move.

**`roc.py`** — predict the output, then run.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
y = torch.tensor([1, 0, 1, 1, 0, 1])
y_proba = torch.tensor([0.9, 0.2, 0.8, 0.7, 0.3, 0.95])
auc = roc_auc_score(y.numpy(), y_proba.numpy())
print(f"AUC-ROC: {auc:.3f}")
# To tune threshold for a specific use case:
fpr, tpr, thresholds = roc_curve(
    y.numpy(), y_proba.numpy())

<details><summary>▸ Expected output</summary>

```
AUC-ROC: 1.000
```
</details>

## 04 · A full evaluation pipeline



<sub>Source: lecture slides 21–24</sub>

**`evaluate.py`** — predict the output, then run.

In [ ]:
def evaluate(model, val_loader, device):
    model.eval()
    all_preds, all_targets, all_probs = [], [], []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = (logits.sigmoid() > 0.5).long()
            all_preds.append(preds)
            all_targets.append(y)
            all_probs.append(logits.sigmoid())
    preds = torch.cat(all_preds)
    targets = torch.cat(all_targets)
    probs = torch.cat(all_probs).cpu()   # every batch
    acc = (preds == targets).float().mean()
    prec = Precision(task="binary")(preds, targets)
    rec = Recall(task="binary")(preds, targets)
    f1 = F1Score(task="binary")(preds, targets)
    auc = roc_auc_score(targets.cpu(), probs)
    print(f"Acc: {acc:.3f} | Prec: {prec:.3f} | "
          f"Rec: {rec:.3f} | F1: {f1:.3f}")
    return {"accuracy": acc, "precision": prec,
            "recall": rec, "f1": f1, "auc": auc}

> ⚠️ **Watch out — Score every batch, not the last one**  
> Metrics are computed once, over the whole validation set — so everything they need must be collected **inside the loop**.
>
> The book’s version takes the AUC’s probabilities from `logits` — the last batch only — and crashes as soon as the loader has more than one batch. The listing above is the corrected one. It assumes the model outputs one logit per sample, shape `(B,)`; a `(B, 1)` output must be squeezed first.

## 05 · Loss curves: a picture tells you why

A number tells you *what*. A picture tells you ***why***.

<sub>Source: lecture slides 27–31</sub>

🧪 *Interactive lab in the app:* **Three loss-curve signatures (plus one)** — try it there to see this idea move.

> ⚠️ **Watch out — Same axes — and log scale**  
> Always plot both curves on the same axes. Two separate plots hide the gap between them, and the gap is exactly what you’re trying to see.
>
> If early losses dwarf later ones, add `plt.yscale("log")` so the late-training detail isn’t squashed flat.

## 06 · Seeing predictions: regression and decision boundaries

Plot the **residuals** (actual − predicted): they should be **shapeless noise around zero**. Any visible pattern is signal your model failed to capture.

<sub>Source: lecture slides 32–39</sub>

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
import matplotlib.pyplot as plt
X_val = torch.linspace(-2, 2, 100).unsqueeze(1)
y_val = 3 * X_val.squeeze() + 0.3 * torch.randn(100)
model = nn.Linear(1, 1)
with torch.no_grad():
    model.weight.fill_(2.8); model.bias.fill_(0.1)


**`pred_vs_actual.py`** — predict the output, then run.

In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(X_val).squeeze()
plt.scatter(y_val, y_pred, s=10, alpha=0.5)
lims = [y_val.min(), y_val.max()]
plt.plot(lims, lims, "r--")                  # perfect prediction
plt.xlabel("actual"); plt.ylabel("predicted")

> ⚠️ **Watch out — Match shapes first**  
> If `y_val` is (N, 1) and `y_pred` is (N,), the subtraction broadcasts to **(N, N)** — the Lecture 06 blow-up. Use `y_val.squeeze() - y_pred`.

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
X = torch.randn(200, 2)
y = (X[:, 0] * X[:, 1] > 0).long()
model = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 2))


**`boundary.py`** — predict the output, then run.

In [ ]:
# Build a grid over the 2-D feature space
xx, yy = torch.meshgrid(
    torch.linspace(-3, 3, 200),
    torch.linspace(-3, 3, 200),
    indexing="xy")
grid = torch.stack([xx.ravel(), yy.ravel()], dim=1)
with torch.no_grad():
    zz = model(grid).argmax(1).reshape(xx.shape)
plt.contourf(xx, yy, zz, alpha=0.3, cmap="coolwarm")
plt.scatter(X[:, 0], X[:, 1], c=y, s=12,
            cmap="coolwarm", edgecolors="k")

## 07 · Seeing your data: image grids and the confusion heatmap

**Look at your images** — before training, after augmentation, and at the mistakes. A wrong normalisation or a mislabelled batch is obvious to the eye and invisible in a loss number.

<sub>Source: lecture slides 40–46</sub>

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
loader = DataLoader(TensorDataset(torch.rand(32, 3, 28, 28), torch.randint(0, 10, (32,))), batch_size=16)
targets = torch.randint(0, 3, (60,))
preds = torch.where(torch.rand(60) < 0.8, targets, torch.randint(0, 3, (60,)))


**`image_grid_heatmap.py`** — predict the output, then run.

In [ ]:
from torchvision.utils import make_grid
imgs, labels = next(iter(loader))                    # (B,C,H,W)
grid = make_grid(imgs[:16], nrow=4,
                 normalize=True)
# make_grid returns (C,H,W); imshow wants (H,W,C)
plt.imshow(grid.permute(1, 2, 0))
plt.axis("off")

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(targets, preds)
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xlabel("predicted"); plt.ylabel("actual")
# Annotate each cell with its count
for i in range(len(cm)):
    for j in range(len(cm)):
        plt.text(j, i, cm[i, j],
                 ha="center", va="center")

## 08 · Looking inside: filters, feature maps, embeddings

Weights are tensors — so **draw them**. Edges and smooth gradients: healthy. Noise: look upstream. This works cleanly for the first layer only; `conv1` is a first convolutional layer (convolutions come in Volume 2).

<sub>Source: lecture slides 47–61</sub>

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.head = nn.Sequential(nn.ReLU(), nn.Flatten(), nn.LazyLinear(10))
    def forward(self, x):
        return self.head(self.conv1(x))
model = TinyCNN()
images = torch.rand(4, 3, 28, 28)


**`filters_hooks.py`** — predict the output, then run.

In [ ]:
# conv1.weight has shape (out, in, kH, kW)
w = model.conv1.weight.detach().cpu()
grid = make_grid(w, nrow=8, normalize=True, padding=1)
plt.imshow(grid.permute(1, 2, 0)); plt.axis("off")

activations = {}
def grab(name):
    def hook(module, inp, out):
        activations[name] = out.detach()
    return hook
# Register once; it fires on every forward pass
handle = model.conv1.register_forward_hook(grab("conv1"))

model.eval()
with torch.no_grad():
    _ = model(images)                      # run a forward pass
maps = activations["conv1"]                # (B, C, H, W)
fmaps = maps[0].unsqueeze(1)               # (C, 1, H, W)
grid = make_grid(fmaps, nrow=8, normalize=True)
plt.imshow(grid.permute(1, 2, 0)); plt.axis("off")
handle.remove()

> ⚠️ **Watch out — Remove the hook**  
> A registered hook stays active until you remove it. `register_forward_hook` returns a handle; call `handle.remove()` when you’re done.
>
> Otherwise it keeps firing on every forward pass — silently slowing training and holding activation tensors that pile up in memory.

## The rule — worth remembering

**Metrics — seven rules**

1. Accuracy is a trap on imbalanced data. Always report precision, recall, or F1.
2. Precision = “of the things I called positive, how many were right?” Use when false positives are costly.
3. Recall = “of the actual positives, how many did I catch?” Use when false negatives are costly.
4. F1 = harmonic mean of precision and recall. A balanced metric for imbalanced problems.
5. Confusion matrix shows all four outcomes. Always visualize it before trusting your metric.
6. For regression: MAE (robust), RMSE (penalizes outliers), R² (explains variance).
7. ROC-AUC measures discrimination threshold-independently. Misleading on extreme imbalance; use PR-AUC instead.

**Visualization — eight rules**

1. Plot train and val loss on the same axes, every run. The gap between them is the diagnosis.
2. Regression: predicted-vs-actual against the diagonal, plus a residual plot that should look like pure noise.
3. 2-D classifiers: shade the decision boundary over a grid to see overfitting as a wiggly border.
4. Images: view them with `make_grid` and remember `permute(1, 2, 0)` for matplotlib.
5. Confusion matrix as a heatmap shows exactly which classes get confused, not just the overall score.
6. Weights and filters are tensors — draw first-layer filters to confirm learning happened.
7. Feature maps come from forward hooks; remember to `remove()` the handle afterward.
8. Embeddings via PCA (fast) or t-SNE (clearer clusters) show whether classes separate. Don’t over-read t-SNE distances.

## Pause and try

Work each exercise in the empty cell below it. Open the worked answer only after you have tried.

### Exercise 01 · Try it — compute by hand

TP = 80, FP = 20, FN = 10, TN = 90. Compute accuracy, precision, recall and F1 by hand. Which metric reveals the biggest flaw in this model?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Accuracy = 170/200 = 0.85. Precision = 80/100 = 0.80. Recall = 80/90 ≈ 0.889. F1 = 2·0.80·0.889 / (0.80 + 0.889) ≈ 0.842. Precision is the lowest: one in five of its positive calls is a false alarm.
</details>

### Exercise 02 · Ch 32 · Ex 1

200 samples, 150 positives and 50 negatives; train a simple logistic model. Compute accuracy, precision, recall and F1. Which best reflects model quality? (Compute specificity too.)

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

With 75% positives, a model that says “positive” to everything already scores 0.75 accuracy, 0.75 precision and perfect recall. **Specificity** — how many of the 50 negatives it actually identifies — is the number that exposes whether it has learned anything.
</details>

### Exercise 03 · Ch 32 · Ex 5 & 6

Sweep the threshold from 0 to 1; plot precision, recall and F1 against it. Where is F1 highest? Then compute AUC-ROC and PR-AUC. Which is higher? Why?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Recall falls and precision rises as the threshold increases; F1 peaks where the two curves cross — often not at 0.5. With a rare positive class, AUC-ROC is usually higher than PR-AUC, because ROC’s false-positive rate is diluted by the many easy negatives. That is why ROC can flatter a model on imbalanced data.
</details>

### Exercise 04 · Ch 33 · Ex 2 & 3

Regression on a noisy quadratic: plot predicted-vs-actual with the y = x line, then a residual plot. Then a classifier on two moons: plot its decision boundary, and retrain a deliberately oversized model.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

A linear model on quadratic data shows a curved band in predicted-vs-actual and a clear U-shape in the residuals — structure the scatter half-hid. The oversized two-moons model draws a jagged boundary that wraps around individual noisy points: overfitting you can see.
</details>

### Exercise 05 · Try it — watch your augmentation

Grab one batch from a DataLoader that applies random augmentation; display it with `make_grid`. Draw a fresh batch and display again. (Use `shuffle=False`, or the fresh batch holds different images.)

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

The same base images appear cropped, flipped or jittered differently the second time — the transform runs on every `__getitem__`.
</details>

### Exercise 06 · Ch 33 · Ex 6 & 7

Hook the first conv layer; display one digit’s feature maps — which channels fire most strongly? Remove the hook when done. Then run t-SNE on the penultimate features of the MNIST test set, coloured by label. Which digits overlap most?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Different channels light up on different strokes and edges of the digit. In the t-SNE plot the overlapping clusters are typically visually similar digits such as 4/9 and 3/5/8 — and those pairs should match the bright off-diagonal cells in your confusion heatmap.
</details>

### Interview-style

(a) “Your model achieves 95% accuracy on a medical screening dataset where 95% of patients are negative, and it predicts everyone as negative. What is its recall? What would you report to a hospital?”

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

Its recall is 0: it catches none of the actual positive cases (TP = 0), and its precision is undefined because it never predicts positive. I would report the confusion matrix alongside recall, precision and F1 — and say plainly that the model is useless for screening despite its accuracy, since a screening tool’s job is to catch cases.
</details>

### Interview-style

(b) “The loss curve looks fine, but accuracy on new data is poor. Walk me through three visualizations you would create, and what each could tell you.” — five to six sentences.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

First, train and validation loss on the same axes: if validation turns upward while training keeps falling, the model is overfitting. Second, a confusion-matrix heatmap on the new data, which shows exactly which classes are being confused rather than one summary number. Third, an image grid of the inputs as the model sees them — after preprocessing — to catch a wrong normalization, a train/inference transform mismatch or mislabeled data. If those look healthy, I’d plot a t-SNE of the penultimate features to see whether the new data’s classes separate at all. Each plot answers a different question: is it memorizing, which mistakes, is the data right, and are the representations meaningful.
</details>